# 신규 애니메이션 수집 (MAL + AniList)

MyAnimeList API에서 시즌별 신규 애니메이션을 조회하고,
AniList API에서 추가 메타데이터(AniList ID, 이미지 등)를 보충하여 DB에 추가합니다.

**워크플로우**
1. MAL 시즌 API로 현재/다음 시즌 anime 목록 조회
2. `anime_external_id.mal_id`로 이미 등록된 anime 제외
3. AniList에서 `idMal` 매칭으로 추가 메타데이터 수집
4. `anime` + `anime_metadata` + `anime_external_id` 테이블에 INSERT

In [10]:
import sys
import time
import json
import dotenv
import os
import psycopg2
from psycopg2.extras import execute_batch

sys.path.insert(0, "scripts")
from external_api import (
    clean_html_synopsis,
    mal_get_season,
    format_air_year_quarter,
    translate_genres,
    anilist_search,
    match_anilist_by_title_and_year,
    ANILIST_FORMAT_MAP,
    MAL_MEDIUM_MAP,
)

dotenv.load_dotenv(dotenv.find_dotenv())

pg = psycopg2.connect(
    host=os.environ["POSTGRES_HOST"],
    port=os.environ["POSTGRES_PORT"],
    dbname=os.environ["POSTGRES_DB"],
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
)
cur = pg.cursor()

MAL_CLIENT_ID = os.environ["MAL_CLIENT_ID"]
print("DB 연결 완료")

DB 연결 완료


## 수집 대상 시즌 설정

In [11]:
# 수집할 시즌 목록 (수동 설정)
# MAL season: winter(1월~), spring(4월~), summer(7월~), fall(10월~)
TARGET_SEASONS = [
    (2026, "spring"),
    (2026, "summer"),
]

print(f"수집 대상 시즌: {TARGET_SEASONS}")

수집 대상 시즌: [(2026, 'spring'), (2026, 'summer')]


In [12]:
# MAL에서 시즌별 anime 목록 조회
mal_anime_list = []

for year, season in TARGET_SEASONS:
    print(f"\n{year} {season} 조회 중...")
    results = mal_get_season(year, season, MAL_CLIENT_ID, limit=500)
    mal_anime_list.extend(results)
    print(f"  {len(results)}개 조회됨")

# mal_id 기준 중복 제거
seen_ids = set()
unique_list = []
for a in mal_anime_list:
    if a["id"] not in seen_ids:
        seen_ids.add(a["id"])
        unique_list.append(a)
mal_anime_list = unique_list

print(f"\n총 MAL anime: {len(mal_anime_list)}개 (중복 제거 후)")


2026 spring 조회 중...
  74개 조회됨

2026 summer 조회 중...
  42개 조회됨

총 MAL anime: 116개 (중복 제거 후)


In [13]:
# 이미 DB에 등록된 mal_id 조회
cur.execute("SELECT mal_id FROM anime_external_id WHERE mal_id IS NOT NULL")
existing_mal_ids = {row[0] for row in cur.fetchall()}
print(f"이미 등록된 MAL ID: {len(existing_mal_ids)}개")

# 미등록 anime만 필터링
new_anime = [a for a in mal_anime_list if a["id"] not in existing_mal_ids]
print(f"신규 anime (미등록): {len(new_anime)}개")

# 미디어 타입 분포
from collections import Counter
type_dist = Counter(a.get("media_type", "unknown") for a in new_anime)
for t, c in type_dist.most_common():
    print(f"  {t}: {c}")

이미 등록된 MAL ID: 1419개
신규 anime (미등록): 0개


## AniList에서 추가 메타데이터 수집

In [14]:
# 각 신규 anime에 대해 AniList에서 추가 메타데이터 수집
enriched = []  # (mal_data, anilist_data or None)

for i, mal_data in enumerate(new_anime):
    if i > 0 and i % 50 == 0:
        matched = sum(1 for _, al in enriched if al is not None)
        print(f"  진행: {i}/{len(new_anime)} (AniList 매칭: {matched})")

    alt = mal_data.get("alternative_titles", {})
    ja_title = alt.get("ja", "")
    en_title = mal_data.get("title", "")
    search_title = ja_title or en_title

    anilist_data = None

    if search_title:
        try:
            candidates = anilist_search(search_title, per_page=5)
            # idMal 정확 매칭 (가장 신뢰도 높음)
            for c in candidates:
                if c.get("idMal") == mal_data["id"]:
                    anilist_data = c
                    break
            # fallback: 제목+연도 매칭
            if not anilist_data:
                year = mal_data.get("start_season", {}).get("year")
                anilist_data = match_anilist_by_title_and_year(
                    candidates, search_title, year
                )
        except Exception:
            pass

    enriched.append((mal_data, anilist_data))
    time.sleep(0.7)  # AniList rate limit: 90 req/min

matched_count = sum(1 for _, al in enriched if al is not None)
print(f"\nAniList 매칭 완료")
print(f"  매칭 성공: {matched_count}/{len(enriched)}")
print(f"  매칭 실패 (MAL 데이터만 사용): {len(enriched) - matched_count}")


AniList 매칭 완료
  매칭 성공: 0/0
  매칭 실패 (MAL 데이터만 사용): 0


## 신규 anime DB 저장

In [15]:
# 신규 anime 레코드 구성 + INSERT
# anime 테이블의 다음 ID를 확보
cur.execute("SELECT COALESCE(MAX(id), 0) FROM anime")
next_id = cur.fetchone()[0] + 1
print(f"새 anime ID 시작: {next_id}")

anime_rows = []       # anime 테이블
metadata_rows = []    # anime_metadata 테이블
external_id_rows = [] # anime_external_id 테이블

for mal_data, anilist_data in enriched:
    anime_id = next_id
    next_id += 1

    alt = mal_data.get("alternative_titles", {})
    ja_title = alt.get("ja", "")
    en_title = mal_data.get("title", "")

    # 이름: MAL 영문 제목 (한국어 소스 없음)
    name = en_title

    # 줄거리: MAL synopsis > AniList description
    synopsis = mal_data.get("synopsis", "")
    if not synopsis and anilist_data and anilist_data.get("description"):
        synopsis = clean_html_synopsis(anilist_data["description"])

    # 제작사
    studios = mal_data.get("studios", [])
    production = studios[0]["name"] if studios else ""
    if not production and anilist_data:
        al_studios = anilist_data.get("studios", {}).get("nodes", [])
        if al_studios:
            production = al_studios[0]["name"]

    # 미디어 타입
    medium = MAL_MEDIUM_MAP.get(mal_data.get("media_type", ""), "")

    # 방영 분기
    air_yq = format_air_year_quarter(mal_data.get("start_season"))

    # 평점: MAL > AniList (AniList는 100점 척도 → 10점으로 변환)
    avg_rating = mal_data.get("mean")
    if avg_rating is None and anilist_data and anilist_data.get("meanScore"):
        avg_rating = anilist_data["meanScore"] / 10.0

    # anime 테이블 row
    anime_rows.append((
        anime_id, name, synopsis, production, medium, air_yq,
        avg_rating, False, False, False,  # is_adult, is_dubbed, is_uncensored
    ))

    # anime_metadata
    genres = translate_genres([g["name"] for g in mal_data.get("genres", [])])
    images = []
    # MAL 이미지
    main_pic = mal_data.get("main_picture", {})
    if main_pic.get("large") or main_pic.get("medium"):
        images.append({
            "optionName": "home_default",
            "imgUrl": main_pic.get("large") or main_pic.get("medium"),
            "cropRatio": "2:3",
        })
    # AniList coverImage 보충
    if anilist_data and not images:
        cover = anilist_data.get("coverImage", {})
        cover_url = cover.get("large") or cover.get("medium")
        if cover_url:
            images.append({
                "optionName": "home_default",
                "imgUrl": cover_url,
                "cropRatio": "2:3",
            })

    production_companies = [{"name": s["name"]} for s in studios]

    metadata_rows.append((
        anime_id,
        json.dumps(genres, ensure_ascii=False),
        json.dumps([], ensure_ascii=False),  # tags
        json.dumps(images, ensure_ascii=False),
        json.dumps([], ensure_ascii=False),  # directors
        json.dumps(production_companies, ensure_ascii=False),
        json.dumps([], ensure_ascii=False),  # casts
    ))

    # anime_external_id
    anilist_id = anilist_data["id"] if anilist_data else None
    anilist_title = anilist_data.get("title", {}).get("romaji", "") if anilist_data else None
    external_id_rows.append((
        anime_id,
        mal_data["id"],
        anilist_id,
        en_title,
        anilist_title,
    ))

print(f"레코드 구성 완료: {len(anime_rows)}개")

새 anime ID 시작: 44044
레코드 구성 완료: 0개


In [16]:
# DB INSERT
execute_batch(cur, """
    INSERT INTO anime (id, name, synopsis, production, medium, air_year_quarter,
                       avg_rating, is_adult, is_dubbed, is_uncensored)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (id) DO NOTHING;
""", anime_rows, page_size=500)

execute_batch(cur, """
    INSERT INTO anime_metadata (anime_id, genres, tags, images, directors, production_companies, casts)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (anime_id) DO NOTHING;
""", metadata_rows, page_size=500)

execute_batch(cur, """
    INSERT INTO anime_external_id (anime_id, mal_id, anilist_id, mal_title, anilist_title)
    VALUES (%s, %s, %s, %s, %s)
    ON CONFLICT (anime_id) DO NOTHING;
""", external_id_rows, page_size=500)

pg.commit()
print(f"INSERT 완료")
print(f"  anime: {len(anime_rows)}")
print(f"  anime_metadata: {len(metadata_rows)}")
print(f"  anime_external_id: {len(external_id_rows)}")

INSERT 완료
  anime: 0
  anime_metadata: 0
  anime_external_id: 0


## 검증

In [17]:
# 결과 확인
cur.execute("SELECT COUNT(*) FROM anime")
total = cur.fetchone()[0]
print(f"전체 anime 수: {total}")

cur.execute("SELECT COUNT(*) FROM anime_external_id WHERE mal_id IS NOT NULL")
mal_count = cur.fetchone()[0]
print(f"MAL 매핑 수: {mal_count}")

cur.execute("SELECT COUNT(*) FROM anime_external_id WHERE anilist_id IS NOT NULL")
anilist_count = cur.fetchone()[0]
print(f"AniList 매핑 수: {anilist_count}")

# 신규 추가 샘플
cur.execute("""
    SELECT a.id, a.name, a.production, a.air_year_quarter, e.mal_id, e.anilist_id
    FROM anime a
    JOIN anime_external_id e ON a.id = e.anime_id
    ORDER BY a.id DESC
    LIMIT 10
""")
print("\n--- 최근 추가 anime 샘플 ---")
for row in cur.fetchall():
    print(f"  [{row[0]}] {row[1]} | {row[2]} | {row[3]} | MAL:{row[4]} | AL:{row[5]}")

전체 anime 수: 8989
MAL 매핑 수: 1419
AniList 매핑 수: 1299

--- 최근 추가 anime 샘플 ---
  [44043] Planosaurus Gachi Koseibutsu-bu | Shogakukan Music & Digital Entertainment | 2026년 3분기 | MAL:63641 | AL:208824
  [44042] Crayon Shin-chan Movie 34: Kikikaikai! Ora no Youkai Bake-shon | Shin-Ei Animation | 2026년 3분기 | MAL:63512 | AL:208829
  [44041] Thunder 3 |  | 2026년 3분기 | MAL:63418 | AL:207254
  [44040] "Kimi wo Aisuru Ki wa Nai" to Itta Jiki Koushaku-sama ga Nazeka Dekiai shitekimasu | Zero-G | 2026년 3분기 | MAL:63537 | AL:208225
  [44039] Chiikawa Movie: Ningyo no Shima no Himitsu | CygamesPictures | 2026년 3분기 | MAL:63011 | AL:202717
  [44038] Kimi to Hanabi to Yakusoku to | SynergySP | 2026년 3분기 | MAL:63366 | AL:206819
  [44037] Reiwa no Dara-san | Asahi Production | 2026년 3분기 | MAL:63082 | AL:203880
  [44036] Rakudai Kenja no Gakuin Musou: Nidome no Tensei, S-Rank Cheat Majutsushi Boukenroku | EMT Squared | 2026년 3분기 | MAL:63508 | AL:208044
  [44035] Kore Kaite Shine | Shin-Ei Animation | 2026년 3

In [18]:
cur.close()
pg.close()
print("Done!")

Done!


In [ ]:
import requests

# 1. 검색 요청
search_url = "https://api.aniplustv.com:3100/search"
search_payload = {
  "params": {
    "userid": "",
    "strFind": "주술회전",
    "gotoPage": 1,
    "token": "MjAyNjAzMTgwODI3Mzk="
  }
}
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json, text/plain, */*"
}

res = requests.post(search_url, json=search_payload, headers=headers)

# 응답 확인
print("검색 응답:", res.status_code)
print("응답 코드:", res.status_code)
print("응답 헤더:", res.headers)
print("응답 원문:", res.text)  # JSONDecodeError 방지용


search_data = res.json()
print(search_data)

# contentSerial 추출
list_data = search_data[0].get("listData", [])
if list_data:
    content_serial = list_data[0]["contentSerial"]
    print("contentSerial:", content_serial)

    # 2. 상세 조회 요청
    detail_url = f"https://api.aniplustv.com:3100/itemInfo?contentSerial={content_serial}"
    detail_res = requests.get(detail_url, headers=headers)

    print("상세 응답:", detail_res.status_code)
    detail_data = detail_res.json()
    print(detail_data)
else:
    print("검색 결과 없음")